# 07. Experiment Summary, Variants and Model Pipeline Selection — `proyecto_integrador_v2`

This notebook summarizes the evolution of the project pipeline versions and documents the final selection.

Includes:

1. **V1**: initial breed classification approach.
2. **V1 adjusted**: improvement of V1 with curation, YOLO/crops and better visual preparation.
3. **V2**: shift toward visual re-identification with embeddings and cosine similarity.
4. **V2 optimized**: final improvement with optimized detection/crop.
5. **Model comparison**: EfficientNetB0 vs EfficientNetV2B0/V2B1.
6. Final technical selection of the recommended pipeline.

The current recommended version is **V2 optimized**:

```python
YOLO_MODEL = "yolo26s.pt"
CONF_THRESHOLD = 0.15
CROP_MARGIN = 0.35
EMBEDDING_BACKBONE = "EfficientNetB0"
SIMILARITY = "cosine"
TOP_K = 10
```

# Version context and adjustment process


This document summarizes the experimental evolution of the visual dog identification project, from an initial breed classification approach to an optimized version oriented toward visual re-identification for lost/found dog cases.

The project started as a breed classification system, but evolved toward a visual search system for lost/found dogs.

The evolution was as follows:

| Version | Objective | Question it answers | Result |
|---|---|---|---|
| **V1** | Classify breed | What breed does this dog appear to be? | Useful, but insufficient to identify lost dogs |
| **V1 adjusted** | Improve visual classification | Can we improve input quality and crop? | Better visual base, but still focused on breed |
| **V2** | Visual re-identification | Does this dog look like one already reported? | Much more aligned to real case |
| **V2 optimized** | Improve coverage and matching | Can we detect more dogs and improve Top-K? | Current recommended version |

To correctly understand the comparisons, it's important to distinguish the evaluated versions:

## V1 — Breed Classification

The first version of the project focused on solving a classification problem: identifying the breed of a dog from an image.

At this stage, the main objective was to train and evaluate models capable of answering:

```text
What breed does this dog appear to be?
```

This version allowed building a first technical base using computer vision models, classification metrics and image preparation. However, an important limitation was identified: recognizing the breed does not equal identifying the dog. Two dogs of the same breed can be completely different individuals, and many lost dogs can be mixed breeds or not clearly correspond to a specific breed.

For that reason, V1 was useful as an academic and technical starting point, but was not sufficient to solve the real case of lost dogs.

## V1 adjusted — Visual preparation improvement

After the initial V1, adjustments were made to improve the visual quality of the images before feeding the models. This stage included curation processes, quality review, use of OpenCV, contrast/sharpness improvements and dog detection/crop using YOLO.

The objective of this adjusted version was to answer:

```text
Can we improve the visual input before classifying or comparing?
```

This version improved dataset cleaning, crop quality and processing traceability. However, although visual input improved, the approach was still primarily related to classification or general visual analysis, not individual re-identification.

Therefore, V1 adjusted was an important transition between breed classification and visual search by identity.

## V2 — Visual re-identification

V2 changed the problem approach. Instead of asking only about breed, the system began to represent each dog as a visual vector or embedding.

The objective became:

```text
Does this dog visually resemble one already reported?
```

In this version, YOLO detects and crops the dog, EfficientNetB0 generates visual embeddings, and cosine similarity allows searching for the most similar neighbors within a vector base.

This change was the most important at a conceptual level, because the lost/found dogs problem is not solved only with breed classification, but with visual recovery of possible matches.

V2 allowed evaluating metrics more aligned with the product, such as:

```text
Top-1 Same Dog Accuracy
Top-5 Same Dog Accuracy
Mean same-dog cosine
Mean different-dog cosine
Separation margin
False positives
False negatives
```

## V2 optimized — Detection, crop and visual search improvement

After validating V2, it was identified that one of the main bottlenecks was not in the embeddings, but in the detection and crop stage. Some images did not generate embeddings because YOLO did not correctly detect the dog.

That's why an optimized configuration was tested:

```python
CONF_THRESHOLD = 0.15
CROP_MARGIN = 0.35
```

This configuration lowers YOLO's confidence threshold to recover more dogs and increases the crop margin to preserve more body and visual context.

V2 optimized consolidated a new line of notebooks:

```text
02B → 03B → 04B → 05B
```

where:

```text
02B = optimized detection/crop
03B = embeddings on optimized crops
04B = optimized vector search
05B = optimized end-to-end test
```

This version increased detection coverage, generated more useful embeddings and improved the main re-identification metric:

```text
Top-5 Same Dog Accuracy
```

## Model comparison

In addition to comparing pipeline versions, different EfficientNet backbones were also tested to evaluate if a newer architecture improved re-identification.

The compared models were:

```text
EfficientNetB0
EfficientNetV2B0
EfficientNetV2B1
```

Although EfficientNetV2B1 achieved better average separation between same dogs and different dogs, EfficientNetB0 obtained better Top-5 recovery, which is the most relevant metric for the use case.

For that reason, the final selection was not based only on the most modern model or with greater average separation, but on the model that best recovers useful candidates for human review.

## General adjustment and testing process

The experimental process followed an incremental logic:

```text
1. Build a first functional version.
2. Measure its results.
3. Identify the main bottleneck.
4. Adjust a pipeline variable.
5. Compare against baseline.
6. Keep the change only if it improves the relevant metric.
7. Consolidate the best configuration in a new version.
```

In this case, the most important change was moving from breed classification to visual re-identification. After that, the most effective improvement was optimizing detection and crop, not changing to a larger model.

The currently selected version is:

```text
V2 optimized with YOLO yolo26s.pt, CONF_THRESHOLD 0.15, CROP_MARGIN 0.35 and EfficientNetB0.
```

This configuration offers the best balance between coverage, visual recovery quality and practical utility for a lost/found dogs system.

In [ ]:
# 1. Imports

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 200)

In [ ]:
# 2. Version comparison: V1, V1 adjusted, V2 and V2 optimized

version_summary_df = pd.DataFrame([
    {
        "version": "V1",
        "main_goal": "Breed classification",
        "core_question": "What breed does this dog appear to be?",
        "main_method": "EfficientNet as breed classifier",
        "main_output": "Predicted breed / Top-5 breeds",
        "strength": "Good academic base for multiclass classification",
        "limitation": "Breed doesn't identify the individual; two dogs of the same breed can be different"
    },
    {
        "version": "V1 adjusted",
        "main_goal": "Improve visual quality for classification",
        "core_question": "Can we improve input before classifying?",
        "main_method": "Visual curation + OpenCV + YOLO crop + EfficientNet",
        "main_output": "Classification with cleaner images/crops",
        "strength": "Better quality control, better crops and more defensible dataset",
        "limitation": "Still primarily solves breed, not identity"
    },
    {
        "version": "V2",
        "main_goal": "Visual re-identification",
        "core_question": "Does this dog visually resemble one already reported?",
        "main_method": "YOLO crop + EfficientNetB0 embeddings + cosine similarity",
        "main_output": "Top-K visual neighbors",
        "strength": "Aligned to real lost/found dogs problem",
        "limitation": "Depended on initial detection/crop configuration"
    },
    {
        "version": "V2 optimized",
        "main_goal": "Visual re-identification with greater coverage",
        "core_question": "Can we recover more dogs without losing matching quality?",
        "main_method": "YOLO conf 0.15 + crop margin 0.35 + EfficientNetB0 embeddings + Top-K",
        "main_output": "Top-K visual candidates + visual decision",
        "strength": "Better coverage, better Top-5 Same Dog Accuracy and fewer false negatives",
        "limitation": "Still requires human review and calibration with more real data"
    },
])

display(version_summary_df)

# Latest applied change

The latest important adjustment was optimizing detection and crop.

Base configuration:

```python
CONF_THRESHOLD = 0.25
CROP_MARGIN = 0.15
```

Optimized configuration:

```python
CONF_THRESHOLD = 0.15
CROP_MARGIN = 0.35
```

This change was consolidated in the optimized line:

```text
02B → 03B → 04B → 05B
```

The reason was simple: lowering the detection threshold allowed recovering more dogs, and increasing the crop margin allowed preserving more body/contextual information useful for re-identification.

In [ ]:
# 3. Detection/crop comparison: V2 base vs V2 optimized

detection_comparison_df = pd.DataFrame([
    {
        "pipeline": "V2 base / Step 02",
        "yolo_model": "yolo26s.pt",
        "conf_threshold": 0.25,
        "crop_margin": 0.15,
        "images_evaluated": 20580,
        "dogs_detected": 19153,
        "not_detected": 1427,
        "detection_rate_percent": 93.07,
        "crops_ok": 19153,
        "crop_errors": 0,
        "avg_yolo_confidence": 0.8455,
        "multiple_dogs_detected_cases": 2316,
    },
    {
        "pipeline": "V2 optimized / Step 02B",
        "yolo_model": "yolo26s.pt",
        "conf_threshold": 0.15,
        "crop_margin": 0.35,
        "images_evaluated": 20580,
        "dogs_detected": 19568,
        "not_detected": 1012,
        "detection_rate_percent": 95.08,
        "crops_ok": 19568,
        "crop_errors": 0,
        "avg_yolo_confidence": 0.8270,
        "multiple_dogs_detected_cases": 2887,
    },
])

detection_comparison_df["extra_crops_vs_base"] = (
    detection_comparison_df["crops_ok"] - detection_comparison_df.loc[0, "crops_ok"]
)

display(detection_comparison_df)

## Reading

V2 optimized recovered **415 additional crops**:

```text
19,153 → 19,568
```

It also reduced cases without detection:

```text
1,427 → 1,012
```

YOLO's average confidence dropped slightly, which is normal when accepting detections with a lower threshold. The result is positive because more useful images were recovered without crop errors.

In [ ]:
# 4. Embeddings comparison: V2 base vs V2 optimized

embedding_comparison_df = pd.DataFrame([
    {
        "pipeline": "V2 base / Step 03",
        "input_report": "step02_detection_report.csv",
        "embedding_model": "EfficientNetB0 ImageNet pooling avg",
        "embedding_dim": 1280,
        "embeddings_generated": 19153,
        "embedding_errors": 0,
        "normalization": "L2",
    },
    {
        "pipeline": "V2 optimized / Step 03B",
        "input_report": "step02b_detection_report_optimized.csv",
        "embedding_model": "EfficientNetB0 ImageNet pooling avg",
        "embedding_dim": 1280,
        "embeddings_generated": 19568,
        "embedding_errors": 0,
        "normalization": "L2",
    },
])

embedding_comparison_df["extra_embeddings_vs_base"] = (
    embedding_comparison_df["embeddings_generated"] - embedding_comparison_df.loc[0, "embeddings_generated"]
)

display(embedding_comparison_df)

In [ ]:
# 5. Vector search comparison: Step 04 vs 04B

search_comparison_df = pd.DataFrame([
    {
        "pipeline": "V2 base / Step 04",
        "embeddings": 19153,
        "avg_top1_similarity": 0.7589,
        "avg_top5_similarity": 0.7227,
        "avg_top10_similarity": 0.7028,
        "median_top1_similarity": 0.7605,
        "top1_ge_0_90": 854,
        "top1_0_80_0_90": 4754,
        "top1_0_70_0_80": 9267,
        "top1_lt_0_70": 4278,
    },
    {
        "pipeline": "V2 optimized / Step 04B",
        "embeddings": 19568,
        "avg_top1_similarity": 0.7489,
        "avg_top5_similarity": 0.7124,
        "avg_top10_similarity": 0.6923,
        "median_top1_similarity": 0.7499,
        "top1_ge_0_90": 819,
        "top1_0_80_0_90": 4165,
        "top1_0_70_0_80": 9374,
        "top1_lt_0_70": 5210,
    },
])

display(search_comparison_df)

## Vector search reading

V2 optimized has more embeddings and covers more difficult cases. That's why average similarity drops slightly.

This doesn't mean the pipeline is worse: it means it now includes images that were previously left out. The most important metric for the product is re-identification, especially:

```text
Top-5 Same Dog Accuracy
```

In [ ]:
# 6. Re-identification comparison: V2 base vs V2 optimized

reid_comparison_df = pd.DataFrame([
    {
        "pipeline": "V2 base",
        "conf_threshold": 0.25,
        "crop_margin": 0.15,
        "valid_crops": 371,
        "no_dog_detected": 157,
        "num_dogs": 97,
        "top1_same_dog_accuracy": 0.9218,
        "top5_same_dog_accuracy": 0.9569,
        "top10_same_dog_accuracy": 0.9596,
        "mean_same_dog_cosine": 0.6835,
        "mean_different_dog_cosine": 0.2261,
        "separation_margin": 0.4574,
        "false_negatives_top5": 16,
        "false_positives_top1": 29,
    },
    {
        "pipeline": "V2 optimized",
        "conf_threshold": 0.15,
        "crop_margin": 0.35,
        "valid_crops": 412,
        "no_dog_detected": 116,
        "num_dogs": 97,
        "top1_same_dog_accuracy": 0.9320,
        "top5_same_dog_accuracy": 0.9709,
        "top10_same_dog_accuracy": 0.9806,
        "mean_same_dog_cosine": 0.6805,
        "mean_different_dog_cosine": 0.2253,
        "separation_margin": 0.4552,
        "false_negatives_top5": 12,
        "false_positives_top1": 28,
    },
    {
        "pipeline": "V2 optimized alternative",
        "conf_threshold": 0.15,
        "crop_margin": 0.25,
        "valid_crops": 412,
        "no_dog_detected": 116,
        "num_dogs": 97,
        "top1_same_dog_accuracy": 0.9296,
        "top5_same_dog_accuracy": 0.9709,
        "top10_same_dog_accuracy": 0.9806,
        "mean_same_dog_cosine": 0.6802,
        "mean_different_dog_cosine": 0.2251,
        "separation_margin": 0.4551,
        "false_negatives_top5": 12,
        "false_positives_top1": 29,
    },
])

for col in ["top1_same_dog_accuracy", "top5_same_dog_accuracy", "top10_same_dog_accuracy"]:
    reid_comparison_df[col + "_percent"] = reid_comparison_df[col] * 100

display(reid_comparison_df)

## Re-identification reading

V2 optimized improved the main metric:

```text
Top-5 Same Dog Accuracy: 95.69% → 97.09%
```

It also improved:

```text
Top-1 Same Dog Accuracy: 92.18% → 93.20%
Top-10 Same Dog Accuracy: 95.96% → 98.06%
False negatives Top-5: 16 → 12
False positives Top-1: 29 → 28
```

The variant `0.15 / 0.35` was slightly better than `0.15 / 0.25`, so it's selected as the recommended configuration.

# EfficientNet Model Comparison

EfficientNet variants were tested to see if a newer architecture improved re-identification.

The comparison showed that EfficientNetV2B1 improves average separation, but doesn't improve the main product metric: **Top-5 Same Dog Accuracy**.

Therefore, the recommended model remains **EfficientNetB0**.

In [ ]:
# 7. EfficientNet backbone comparison

model_comparison_df = pd.DataFrame([
    {
        "backbone": "EfficientNetB0",
        "top1_same_dog_accuracy": 0.9218,
        "top5_same_dog_accuracy": 0.9569,
        "top10_same_dog_accuracy": 0.9596,
        "mean_same_dog_cosine": 0.6835,
        "mean_different_dog_cosine": 0.2261,
        "separation_margin": 0.4574,
        "false_negatives_top5": 16,
        "false_positives_top1": 29,
        "decision": "Main recommended model"
    },
    {
        "backbone": "EfficientNetV2B0",
        "top1_same_dog_accuracy": 0.9084,
        "top5_same_dog_accuracy": 0.9542,
        "top10_same_dog_accuracy": 0.9569,
        "mean_same_dog_cosine": 0.7005,
        "mean_different_dog_cosine": 0.2193,
        "separation_margin": 0.4811,
        "false_negatives_top5": 17,
        "false_positives_top1": 34,
        "decision": "Doesn't improve Top-5; not recommended as main"
    },
    {
        "backbone": "EfficientNetV2B1",
        "top1_same_dog_accuracy": 0.9218,
        "top5_same_dog_accuracy": 0.9515,
        "top10_same_dog_accuracy": 0.9569,
        "mean_same_dog_cosine": 0.7103,
        "mean_different_dog_cosine": 0.2152,
        "separation_margin": 0.4951,
        "false_negatives_top5": 18,
        "false_positives_top1": 29,
        "decision": "Better separation, but worse Top-5 than B0"
    },
])

for col in ["top1_same_dog_accuracy", "top5_same_dog_accuracy", "top10_same_dog_accuracy"]:
    model_comparison_df[col + "_percent"] = model_comparison_df[col] * 100

display(model_comparison_df)

## Model reading

EfficientNetV2B1 achieved better average separation:

```text
Separation margin: 0.4951
```

but EfficientNetB0 had better Top-5 recovery:

```text
EfficientNetB0 Top-5: 95.69%
EfficientNetV2B1 Top-5: 95.15%
```

Since the product objective is to show correct candidates for human review, the most important metric is **Top-5 Same Dog Accuracy**.  
Therefore **EfficientNetB0** is selected as the main backbone.

# Final pipeline selection

The recommended version is:

```text
V2 optimized
```

Final configuration:

```python
YOLO_MODEL = "yolo26s.pt"
CONF_THRESHOLD = 0.15
CROP_MARGIN = 0.35
EMBEDDING_BACKBONE = "EfficientNetB0"
EMBEDDING_DIM = 1280
NORMALIZATION = "L2"
SIMILARITY = "cosine"
TOP_K = 10
```

Final pipeline:

```text
01_Image_Quality_Curation
↓
02B_Dog_Detection_Cropping_YOLO_Optimized
↓
03B_Dog_Visual_Embeddings_Optimized
↓
04B_Vector_Search_Cosine_Similarity_Optimized
↓
05B_Lost_Found_End_to_End_Search_Optimized
↓
06_Dog_ReIdentification_Evaluation
↓
07_Experiment_Summary_and_Model_Selection
```

In [ ]:
# 8. Configuración final seleccionada

final_selection_df = pd.DataFrame([
    {"component": "Pipeline version", "selected_value": "V2 optimizada"},
    {"component": "YOLO model", "selected_value": "yolo26s.pt"},
    {"component": "Confidence threshold", "selected_value": 0.15},
    {"component": "Crop margin", "selected_value": 0.35},
    {"component": "Embedding backbone", "selected_value": "EfficientNetB0"},
    {"component": "Embedding dimension", "selected_value": 1280},
    {"component": "Normalization", "selected_value": "L2"},
    {"component": "Similarity metric", "selected_value": "cosine similarity"},
    {"component": "Retrieval", "selected_value": "Top-K neighbors"},
    {"component": "Recommended Top-K", "selected_value": 10},
    {"component": "Main product metric", "selected_value": "Top-5 Same Dog Accuracy"},
])

display(final_selection_df)

,component,selected_value
0,Pipeline version,V2 optimizada
1,YOLO model,yolo26s.pt
2,Confidence threshold,0.15
3,Crop margin,0.35
4,Embedding backbone,EfficientNetB0
5,Embedding dimension,1280
6,Normalization,L2
7,Similarity metric,cosine similarity
8,Retrieval,Top-K neighbors
9,Recommended Top-K,10


# Limitaciones

Aunque la V2 optimizada es la versión recomendada, todavía existen limitaciones:

1. Los matches visuales no deben interpretarse como certeza absoluta.
2. El sistema debe mostrar candidatos para revisión humana.
3. Las imágenes con múltiples perros requieren manejo especial.
4. La detección puede fallar en perros pequeños, parciales o con mala iluminación.
5. El dataset de identidad todavía debe crecer para validar generalización.
6. La similitud visual debe combinarse con metadata real: ubicación, fecha, tamaño, color, descripción y contexto del reporte.

# Próximos pasos recomendados

Los siguientes pasos serían:

1. **Agrupar resultados por `dog_id`** en lugar de mostrar solo vecinos individuales.
2. Implementar **multi-embedding por perro**: varias fotos por identidad.
3. Calibrar umbrales reales de match fuerte / posible / débil.
4. Revisar manualmente falsos positivos y falsos negativos.
5. Agregar metadata contextual: ubicación, fecha, color, collar, tamaño, descripción.
6. Probar modelos especializados para embeddings: DINOv2, CLIP o Siamese/Triplet.
7. Entrenar una red Siamese/Triplet cuando haya suficientes identidades reales.
8. Construir demo web/app para flujo lost/found.

# Conclusión final

La evolución del proyecto muestra que la solución más adecuada no es clasificar razas, sino hacer re-identificación visual.

La **V1** fue útil como base de clasificación.  
La **V1 ajustada** mejoró la calidad visual y el procesamiento de imágenes.  
La **V2** replanteó correctamente el problema hacia embeddings y similitud visual.  
La **V2 optimizada** mejoró la cobertura y el desempeño de re-identificación.

Por lo tanto, la versión recomendada actual es:

```text
V2 optimizada con YOLO yolo26s.pt, CONF_THRESHOLD 0.15, CROP_MARGIN 0.35 y EfficientNetB0.
```

Esta versión ofrece el mejor balance entre cobertura, recuperación visual y utilidad práctica para un sistema de perros perdidos/encontrados.